<a href="https://colab.research.google.com/github/Parthav47/llm_calling/blob/main/CLI_Groq_Web_summarier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## JSON based Groq Web Page summariser

In [ ]:
#!/usr/bin/env python3
import sys
import subprocess


def bootstrap():
    """Installs required libraries before the rest of the script imports them."""
    required = ["groq", "beautifulsoup4", "lxml", "requests"]
    # Check if we are in a virtual environment or need --user
    print("Checking environment dependencies...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *required, "-q"])
    except Exception as e:
        print(f"Error installing dependencies: {e}")
        sys.exit(1)

bootstrap()

import os
import time
import json
import getpass
import requests
from bs4 import BeautifulSoup
from groq import Groq

def scrape_text(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "lxml")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        return soup.get_text(separator=" ", strip=True)[:5000]
    except Exception as e:
        return {"error": f"Scraping failed: {str(e)}"}

def extract_info(client, text, model_name):
    if isinstance(text, dict) and "error" in text:
        return text

    time.sleep(2)
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are an extraction engine. Reply ONLY in JSON."},
                {"role": "user", "content": f"Extract info from this text: {text}"}
            ],
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"error": f"LLM Processing failed: {str(e)}"}

def main():
    print("\n" + "="*30)
    print("  AI TERMINAL SUMMARIZER")
    print("="*30)

    api_key = getpass.getpass("Enter GROQ_API_KEY (hidden): ")
    model_name = input("Enter Model (e.g. llama-3.3-70b-versatile): ")

    try:
        client = Groq(api_key=api_key)
    except Exception as e:
        print(f"Failed to initialize client: {e}")
        return

    while True:
        url = input("\nURL to summarize (or 'exit'): ").strip()
        if url.lower() == 'exit': break
        if not url: continue

        print("Processing...")
        content = scrape_text(url)
        if isinstance(content,dict) and "error" in content:
            print(content["error"])
            return

        result = extract_info(client, content, model_name)
        if isinstance(result,dict) and "error" in result:
            print(result["error"])
            return

        print("\n" + json.dumps(result, indent=2))

if __name__ == "__main__":
    main()